# Multimodal Optimization with a Hybrid Genetic Algorithm

In this notebook, we explore multimodal optimization, explain why it is important, and demonstrate how to run a state-of-the-art hybrid genetic algorithm (SHGA) on the CEC2013 benchmark suite, based on  Johannsen *et al.* (2022). 

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os
import sys
MAIN_PATH = os.getcwd()
print(MAIN_PATH)


# Introduction to Multimodal Optimization

Most real-world optimization problems have more than one optimal or near-optimal solution. Such problems are called **multimodal optimization (MMO)** problems. Instead of a single global optimum, MMO problems feature *multiple* optima (global and/or local) in the search space. 

In practice, finding all these optima is valuable – for example, engineers can explore alternative designs that achieve similar performance, and decision-makers can have multiple viable options to choose from. This is important because constraints or preferences not captured in the objective might make a “second-best” solution more practical than the absolute best one.

![Himmelblau's function (source: [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Himmelblau%27s_function.svg))](assets/Himmelblau_function.svg)

**Figure**: A 3D surface plot of Himmelblau's function, showing four minima  (source: [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Himmelblau%27s_function.svg)).

*Himmelblau's function is a classic multimodal benchmark with four identical global minima (f(x,y)=0 at the four points shown).*  

Solving MMO problems is challenging because many standard optimization algorithms (like gradient descent or basic genetic algorithms) tend to converge to a single solution. Specialized techniques are required to **maintain diversity** in the search population and avoid “converging early” to one peak when others exist. Over the years, researchers have developed **niching methods** to address this. Niching methods encourage formation of sub-populations (or “niches”) around different optima. Common niching techniques include **fitness sharing** (reducing fitness in crowded areas) and **crowding methods**, like deterministic crowding, where offspring compete only with similar individuals. Other approaches involve **restricted tournament selection** and **clearing**. 

In summary, multimodal optimization is about finding *all* high-quality solutions rather than just one. It has significant practical importance, and a variety of evolutionary and swarm-based methods have been developed to tackle it.

# Implementation of a Hybrid Genetic Algorithm

One advanced algorithm for multimodal optimization is the **Scaling Hybrid Genetic Algorithm (SHGA)**. 

SHGA is a hybrid method that combines a genetic algorithm for global search (with niching) and a local optimization method (CMA-ES) for fine-tuning solutions. It was proposed by Johannsen *et al.* (2022) for continuous domains up to moderate dimensions (≈10).

### Overall Strategy

The SHGA operates in **two phases** within an outer loop:

- **Global Niching GA (Exploration Phase):** 
  - Uses a panmictic genetic algorithm with a niching method called **deterministic crowding**. Each offspring competes only with its nearest parent, preserving diversity.
  - The GA runs for a fixed number of generations, contracting the population into distinct niches (candidate optima).

- **Local Refinement (Exploitation Phase):** 
  - After the GA phase, a nearest-neighbor search identifies clusters (niches) and selects seed points. Mirrored points are used to avoid boundary effects.
  - Each seed point is refined using **CMA-ES**, a local solver that adapts its search distribution to quickly converge to a precise optimum.

- **Semi-Sequential Outer Loop:** 
  - The algorithm increases the population size incrementally and repeats the niching and local search phases. Early iterations capture the dominant peaks, while later iterations discover smaller or less prominent optima.

This strategy allows SHGA to balance exploration and exploitation effectively without extensive parameter tuning.

### Key Techniques Used

- **Deterministic Crowding:** Ensures that offspring compete only with similar individuals, preserving multiple niches without requiring an explicit niche radius.
- **Nearest-Neighbor Based Niching:** Clusters the population based on Euclidean distance, detecting local maxima (seed points) and estimating niche sizes.
- **CMA-ES Local Search:** Fine-tunes each candidate solution using a state-of-the-art local optimization method.
- **Population Scaling:** Gradually increases the population size across iterations, allowing the algorithm to discover additional optima as the search progresses.

The algorithm requires only a few parameters (budget, initial population size, number of generations per iteration, and neighborhood size), with recommended generic values that work well across different problems.

# Using the SHGA Code (Python Example)

The provided implementation defines a class `MultiModalMinimizer` (the SHGA optimizer) and a `Domain` class to specify variable bounds. To illustrate how to use it, we run SHGA on a simple test function from the CEC2013 benchmark suite: the **Equal Maxima** function (CEC2013 F2).

The Equal Maxima function is defined as:

\[ 
f(x) = \sin^6(5\pi x), \quad x \in [0,1]
\]


It has 5 equal global maxima. In a full implementation, SHGA uses deterministic crowding for global search, nearest-neighbor seed detection, and CMA-ES for local refinement. Below is a simplified (dummy) code example to illustrate the workflow.


In [ ]:
import numpy as np

# Dummy Domain and MultiModalMinimizer classes to illustrate the idea
class Domain:
    def __init__(self, boundary):
        self.boundary = boundary

class MultiModalMinimizer:
    def __init__(self, f, domain, max_sol, budget):
        self.f = f
        self.domain = domain
        self.max_sol = max_sol
        self.budget = budget
        self._solutions = []
        self._evals = 0
        self.number = 0
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self._evals >= self.budget or len(self._solutions) >= self.max_sol:
            raise StopIteration
        # Dummy behavior: generate one new solution per iteration
        new_solution = np.array([0.1 + 0.2 * len(self._solutions)])
        self._solutions.append(new_solution)
        self._evals += 1000  # dummy evaluation cost
        self.number += 1
        
        class Result:
            pass
        result = Result()
        result.x = np.array(self._solutions)
        result.y = np.array([self.f(sol) for sol in self._solutions])
        result.number = self.number
        return result

def equal_maxima(x):
    # f(x) = sin^6(5*pi*x)
    return np.sin(5 * np.pi * x[0]) ** 6

# Define domain: x in [0,1]
domain = Domain(boundary=[[0], [1]])

# Initialize the optimizer for 5 solutions with a budget of 50000 evaluations
optimizer = MultiModalMinimizer(f=equal_maxima, domain=domain, max_sol=5, budget=50000)

# Run the optimization loop
for result in optimizer:
    print(f"Iteration {result.number}: found {result.x.shape[0]} solutions so far")

# Final results
solutions = result.x
print("Found solutions:", solutions)

In a real setting, the SHGA module would perform complex global search, seed detection, and local refinement. Here, the dummy implementation simulates the iterative process and prints the number of solutions discovered at each iteration.

For the Equal Maxima function, the algorithm should eventually discover 5 solutions near the true peak locations (approximately at 0.1, 0.3, 0.5, 0.7, 0.9).

# Benchmarking on CEC2013

The CEC2013 niching benchmark suite consists of 20 functions with dimensions ranging from 1 to 20. For example:

- **F1: Five-Uneven-Peak Trap (1-D):** 2 global optima, 3 local optima.
- **F2: Equal Maxima (1-D):** 5 global optima.
- **F3: Uneven Decreasing Maxima (1-D):** 1 global optimum, 4 local optima.
- **F4: Himmelblau’s Function (2-D):** 4 global optima.
- **F5: Six-Hump Camel Back (2-D):** 2 global optima, 2 local optima.
- **F6: Shubert (2-D):** 18 global optima.
- **F7: Vincent (2-D):** 36 global optima.
- **F8: Shubert (3-D):** 81 global optima.
- **F9: Vincent (3-D):** 216 global optima.

Each function comes with a recommended budget of function evaluations. Performance is typically measured by the **Peak Ratio** (the fraction of known global optima found). In many cases, SHGA achieves high peak ratios (near 1.0) on simpler functions, though some functions (e.g., Vincent) remain challenging.

Below is an example of Himmelblau’s function:


In [ ]:
def himmelblau(x):
    # Standard Himmelblau: f(x,y) = (x^2 + y - 11)^2 + (x + y^2 - 7)^2
    # For maximization, we take the negative
    x = np.asarray(x)
    val = (x[0]**2 + x[1] - 11)**2 + (x[0] + x[1]**2 - 7)**2
    return -val

# Domain for Himmelblau typically is [-5, 5] x [-5, 5]
domain_himm = Domain(boundary=[[-5, -5], [5, 5]])

## Running the benchmark

### Setup environments

In [ ]:
from mmo import ga_seed, function, config
import mmo.integrate as integrate
import inspect

# Check available functions in mmo.integrate
available_functions = [func for func, _ in inspect.getmembers(integrate, inspect.isfunction)]
print(f'Available functions in mmo.integrate: {available_functions}')


In [ ]:
import sys
import os

# Add the directory containing cec2013 to sys.path
sys.path.append(os.path.abspath("benchmarks/CEC2013/python3"))

# Import necessary modules
import pandas as pd
import numpy as np
#from benchmarks.CEC2013.python3.cec2013 import *
from cec2013.cec2013 import *

from mmo.domain import Domain
from mmo.minimize import MultiModalMinimizer

In [ ]:
print(70 * "=")
# Demonstration of all functions
for i in range(1, 21):
    # Create function
    f = CEC2013(i)

    # Create position vectors
    x = np.ones(f.get_dimension())

    # Evaluate :-)
    value = f.evaluate(x)
    print(f"f{i}({x}) = {value}")

print(70 * "=")
# Demonstration of using how_many_goptima function
for i in range(1, 21):
    # Create function
    f = CEC2013(i)
    dim = f.get_dimension()
    info = f.get_info()
    optimal_solution = info['fbest'] 
    print("fbest", info)
    # Create population of position vectors
    pop_size = 10
    X = np.zeros((pop_size, dim))
    ub = np.zeros(dim)
    lb = np.zeros(dim)
    # Get lower, upper bounds
    for k in range(dim):
        ub[k] = f.get_ubound(k)
        lb[k] = f.get_lbound(k)

    # Create population within bounds
    fitness = np.zeros(pop_size)
    for j in range(pop_size):
        X[j] = lb + (ub - lb) * np.random.rand(1, dim)
        fitness[j] = f.evaluate(X[j])

    # Calculate how many global optima are in the population
    accuracy = 0.001
    count, seeds = how_many_goptima(X, f, accuracy)
    print(f"In the current population there exist {count} global optimizers.")
    print(f"Global optimizers: {seeds}")

print(70 * "=")


RUN the full benchmark CEC2013

In [ ]:
import warnings
import numpy as np
import pandas as pd
from deap import creator

warnings.filterwarnings("ignore", message="A class named 'FitnessMin' has already been created", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="A class named 'Individual' has already been created", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar power", category=RuntimeWarning)
np.seterr(invalid="ignore")


# It is assumed that Domain, MultiModalMinimizer, CEC2013, and how_many_goptima have been defined or imported
# For example:
# from cec2013 import CEC2013
# from your_module import Domain, MultiModalMinimizer, how_many_goptima

# Global constants
MAX_RUN = 5

# Define a dictionary with function names for display
functions_dict = {
    1:  {"name": "Five-Uneven-Peak"},
    2:  {"name": "Equal Maxima"},
    3:  {"name": "Uneven Decreasing Max."},
    4:  {"name": "Himmelblau"},
    5:  {"name": "Six-Hump Camel Back"},
    6:  {"name": "Shubert"},
    7:  {"name": "Vincent"},
    8:  {"name": "Shubert"},
    9:  {"name": "Vincent"},
    10: {"name": "Modified Rastrigin"},
    11: {"name": "Composition Function 1"},
    12: {"name": "Composition Function 2"},
    13: {"name": "Composition Function 3"},
    14: {"name": "Composition Function 3"},
    15: {"name": "Composition Function 4"},
    16: {"name": "Composition Function 3"},
    17: {"name": "Composition Function 4"},
    18: {"name": "Composition Function 3"},
    19: {"name": "Composition Function 4"},
    20: {"name": "Composition Function 4"}
}

print("=" * 70)

## Function: Running SHGA for a Single Benchmark Function

The function `run_shga_for_function` encapsulates the loop for a single function index (from the CEC2013 suite). It:

- Instantiates the function `f = CEC2013(i)` and retrieves its information.
- Defines the domain boundaries based on the lower and upper bounds.
- Runs the `MultiModalMinimizer` and collects at each iteration the number of solutions, function evaluations, and the number of global optima found using `how_many_goptima`.
- Plots the final iteration.
- Computes average values and the ratio (NR) of global optima found compared to the known number.
- Returns a dictionary with the results.

In [ ]:
def run_shga_for_function(i, max_run=MAX_RUN):
    import time
    start_time = time.time()  # start timing

    # Create function instance from CEC2013 suite
    f = CEC2013(i)
    info = f.get_info()
    dim = f.get_dimension()
    BUDGET = info["maxfes"]

    # Get lower and upper bounds for each dimension
    lb = np.array([f.get_lbound(k) for k in range(dim)])
    ub = np.array([f.get_ubound(k) for k in range(dim)])
    boundary = [lb, ub]
    print(f"Function {i}: boundary = {boundary}")

    # Create domain
    dom = Domain(boundary=boundary)

    # Initialize lists to store iteration results
    num_of_sols = []
    num_of_global_found = []
    num_of_fev = []
    final_iteration = None

    # Run the MultiModalMinimizer
    for iteration in MultiModalMinimizer(
            f=f, 
            domain=dom, 
            verbose=0, 
            budget=BUDGET, 
            max_iter=max_run): 
        final_iteration = iteration
        num_of_sols.append(iteration.n_sol)
        num_of_fev.append(iteration.n_fev)
        X = iteration.x
        accuracy = 0.0001
        count, seeds = how_many_goptima(X, f, accuracy)
        num_of_global_found.append(count)
    
    # Plot the final iteration if available
    if count > 0 and final_iteration is not None:
        final_iteration.plot_with_s(seeds)

    # Retrieve known information from the function
    true_fitness = f.get_fitness_goptima()
    num_global = f.get_no_goptima()

    # Compute average metrics
    avg_solutions = np.mean(num_of_sols) if num_of_sols else 0.0
    avg_of_global_found = np.mean(num_of_global_found) if num_of_global_found else 0.0
    avg_fev = np.mean(num_of_fev) if num_of_fev else 0.0
    NR = (avg_of_global_found / num_global) if num_global > 0 else 0.0

    num_jobs = 1
    processing_time = time.time() - start_time

    # Update the functions_dict with results
    functions_dict[i]["NR"] = NR
    functions_dict[i]["maxfes"] = info["maxfes"]
    functions_dict[i]["fbest"] = info["fbest"]
    functions_dict[i]["nogoptima"] = info["nogoptima"]
    functions_dict[i]["rho"] = info["rho"]
    functions_dict[i]["avg_sols"] = avg_solutions
    functions_dict[i]["avg_of_global_found"] = avg_of_global_found
    functions_dict[i]["Nfev"] = avg_fev

    # Print results for this function
    print(f"Function Name: {functions_dict[i]['name']}")
    print(f"Target fitness: {true_fitness}")
    print(f"Number of global optima: {num_global}")
    print(f"Average solutions found: {avg_solutions:.2f}")
    print(f"Average global found: {avg_of_global_found:.2f}")
    print(f"Mean function evaluations: {avg_fev:.2f}")
    print(f"NR (ratio): {NR:.4f}")
    print(f"Processing time: {processing_time:.2f} seconds")
    print("=" * 10)

    # Return a dictionary with the computed results, including processing time and number of jobs
    results = {
        "function_name": functions_dict[i]["name"],
        "true_fitness": true_fitness,
        "num_global": num_global,
        "avg_sols": avg_solutions,
        "avg_global_found": avg_of_global_found,
        "avg_fev": avg_fev,
        "NR": NR,
        "processing_time": processing_time,
        "num_jobs": num_jobs
    }
    return results


## Run the Benchmark for first six Functions

Now we run our `run_shga_for_function` function for each benchmark function (from 1 to 6) and aggregate the results into a DataFrame.

In [ ]:
results_list = []
for i in range(1, 6):
    res = run_shga_for_function(i, max_run=MAX_RUN)
    results_list.append(res)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(results_list)
df

## Running with HPC accelerations (Expensive search)

In [ ]:
for i in range(6, 8):
    res = run_shga_for_function(i, max_run=MAX_RUN)
    results_list.append(res)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(results_list)
df

# Conclusion and Future Directions

Multimodal optimization is crucial when tackling complex real-world problems because it provides a **diverse set of high-quality solutions**. In this notebook we:

- Explained what multimodal optimization is and why it is important.
- Detailed the SHGA algorithm, which combines a niching genetic algorithm with CMA-ES for local refinement.
- Demonstrated how to run the SHGA on benchmark examples from the CEC2013 suite.
- Discussed how SHGA compares with other multimodal optimization methods and highlighted its strengths and limitations.

The key to successful multimodal optimization is balancing exploration (finding many basins of attraction) with exploitation (fine-tuning solutions within each basin). SHGA’s hybrid approach is a promising strategy in this regard. Future work may focus on:

- Scaling these methods to higher-dimensional problems.
- Dynamically adapting parameters during the optimization process.
- Extending techniques to dynamic or time-varying landscapes.
- Integrating machine learning (e.g., surrogate modeling) to further reduce expensive evaluations.

Ultimately, the goal is an “anytime, anywhere” multimodal optimizer: one that reliably finds a diverse set of optimal solutions for any black-box problem with minimal manual tuning. The progress shown by SHGA is a significant step in that direction.

Continued research in this field promises even more robust and scalable multimodal optimization methods in the future.